In [1]:
"""
1. Linear vs Non Linear Regression
2. Support Vectors
3. Kernel Trick
4. Hyperparameter Tuning
5. C
6. Gamma
7. Epsilon
8. Grid Search
9. Randomized Search
10. Train/Validation/Test Workflow
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)

In [2]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

housing = fetch_california_housing()
df = pd.DataFrame(housing.data,columns=housing.feature_names)
df["Target"] = housing.target

print(df.head())
print(df.info())
print(df.describe())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  Target  
0    -122.23   4.526  
1    -122.22   3.585  
2    -122.24   3.521  
3    -122.25   3.413  
4    -122.25   3.422  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOc

In [3]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df.corr())

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
Target        0
dtype: int64
Duplicates: 0
              MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  \
MedInc      1.000000 -0.119034  0.326895  -0.062040    0.004834  0.018766   
HouseAge   -0.119034  1.000000 -0.153277  -0.077747   -0.296244  0.013191   
AveRooms    0.326895 -0.153277  1.000000   0.847621   -0.072213 -0.004852   
AveBedrms  -0.062040 -0.077747  0.847621   1.000000   -0.066197 -0.006181   
Population  0.004834 -0.296244 -0.072213  -0.066197    1.000000  0.069863   
AveOccup    0.018766  0.013191 -0.004852  -0.006181    0.069863  1.000000   
Latitude   -0.079809  0.011173  0.106389   0.069721   -0.108785  0.002366   
Longitude  -0.015176 -0.108197 -0.027540   0.013344    0.099773  0.002476   
Target      0.688075  0.105623  0.151948  -0.046701   -0.024650 -0.023737   

            Latitude  Longitude    Target  
MedInc     -0.

In [4]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("Target", axis=1)
y = df["Target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(14447, 8)
(3097, 8)
(3096, 8)


In [5]:
# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# SVR is distance based so Scaling is mandatory.
pipe = Pipeline([("scaler",StandardScaler()),("model",SVR(kernel="rbf",C=1.0,gamma="scale",epsilon=0.1))])

In [6]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

pipe.fit(X_train,y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0


In [7]:
# =====================================================
# STEP 6 : VALIDATION with base model
# =====================================================

y_pred = pipe.predict(X_val)
mae = mean_absolute_error(y_val,y_pred)
rmse = np.sqrt(mean_squared_error(y_val,y_pred))
r2 = r2_score(y_val,y_pred)

print("\nBASELINE SVR")
print("mae: ",mae)
print("rmse: ",rmse)
print("r2 score: ",r2)



BASELINE SVR
mae:  0.3931182163428107
rmse:  0.5874766950985205
r2 score:  0.7420249816964113


In [8]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(pipe,X_train,y_train,cv=5,scoring="r2",n_jobs=-1)
print("\nCV Mean:", cv_scores.mean())
print("CV Std :", cv_scores.std())



CV Mean: 0.731993497560664
CV Std : 0.019966048529748523


In [ ]:
# =====================================================
# STEP 8 : GRID SEARCH
# =====================================================

param_grid = {
    "model__kernel":["linear","rbf"],
    "model__C":[0.1,1,10,100],
    "model__gamma":["scale","auto",0.01,0.1],
    "model__epsilon":[0.01,0.1,0.5]}

grid = GridSearchCV(pipe,param_grid,cv=3,scoring="r2",n_jobs=-1)
grid.fit(X_train,y_train)

print(grid.best_params_)
print(grid.best_score_)
best_model = grid.best_estimator_
print(best_model)

# =====================================================
# STEP 8.5 : RANDOMIZED SEARCH
# =====================================================

# Real world:
# Huge parameter space
# Use RandomizedSearchCV

random_search = RandomizedSearchCV(pipe,param_distributions=param_grid,n_iter=20,cv=3,scoring="r2",random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)

print("\nRandom Search Best")
print(random_search.best_params_)


In [ ]:
# =====================================================
# STEP 9 : VALIDATION with best model
# =====================================================

val_pred = best_model.predict(X_val)
val_r2 = r2_score(y_val,val_pred)
print("Validation R2:",val_r2)


In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_pred = best_model.predict(X_train)
train_r2 = r2_score(y_train,train_pred)
print("\nTrain R2:", train_r2)
print("Validation R2:", val_r2)



In [ ]:
# =====================================================
# STEP 11 : SVR SPECIFIC ANALYSIS
# =====================================================

svr_model = best_model.named_steps["model"]

# ----------------------------------------------
# SUPPORT VECTORS
# ----------------------------------------------

print("\nNumber of Support Vectors")
print(len(svr_model.support_))

# Interview:
# Support vectors are the points
# closest to decision boundary
# that define the regression function.

# ----------------------------------------------
# Kernel Comparison
# ----------------------------------------------

kernels = ["linear","rbf","poly"]
scores = []

for kernel in kernels:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel=kernel,C=10))])

    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)

plt.bar(kernels,scores)

plt.title("Kernel Comparison")
plt.ylabel("Validation R2")
plt.show()

# ----------------------------------------------
# C Sensitivity
# ----------------------------------------------

c_values = [0.1,1,10,100]
scores = []

for c in c_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf",C=c))])
    
    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)
plt.plot(c_values,scores,marker="o")

plt.xscale("log")
plt.title("C Sensitivity")
plt.xlabel("C")
plt.ylabel("Validation R2")
plt.show()

# ----------------------------------------------
# Gamma Sensitivity
# ----------------------------------------------

gamma_values = [0.001,0.01,0.1,1]
scores = []

for gamma in gamma_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf",gamma=gamma))])

    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)

plt.plot(gamma_values,scores,marker="o")
plt.xscale("log")
plt.title("Gamma Sensitivity")
plt.xlabel("Gamma")
plt.ylabel("Validation R2")
plt.show()



In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model



In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test,test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test,test_pred))
test_r2 = r2_score(y_test,test_pred)

print("\nTEST RESULTS")
print("MAE :", test_mae)
print("RMSE:", test_rmse)
print("R2 :", test_r2)



In [ ]:
# =====================================================
# STEP 14 : INTERPRETATION
# =====================================================

print("\nFINAL MODEL")
print(final_model.named_steps["model"])



In [ ]:
# =====================================================
# STEP 15 : DEPLOYMENT READINESS
# =====================================================

print("\nDEPLOYMENT READINESS")
print("CV Mean:",cv_scores.mean())
print("Validation R2:",val_r2)
print("Test R2:",test_r2)



In [ ]:
# =====================================================
# INTERVIEW QUESTIONS
# =====================================================

"""
1. What is Support Vector Regression?
2. Difference between Linear Regression and SVR?
3. What is Kernel Trick?
4. Why RBF Kernel?
5. What is Support Vector?
6. What is C?
7. What happens if C becomes large?
8. What happens if C becomes small?
9. What is Gamma?
10. What happens when Gamma increases?
11. What is Epsilon?
12. What is Epsilon Tube?
13. Why scaling is mandatory?
14. GridSearchCV vs RandomizedSearchCV?
15. Why does SVR become slow on large datasets?
16. Difference between SVC and SVR?
17. Which kernels are commonly used?
18. When would you choose SVR over Linear Regression?
19. What causes overfitting in SVR?
20. How do you tune SVR?
"""